# GIF625 Analytic HG Exercises

This notebook sets up a 15-mode GIF625-like scalar multimode propagation problem using analytic separable Hermite-Gaussian transverse modes. The dispersion coefficients are loaded from the public scalar GIF625 CSV table, while the Kerr overlap tensor is built analytically in the HG basis.

The first analytic modes are ordered as `(0,0)`, `(1,0)`, `(0,1)`, `(2,0)`, `(1,1)`, `(0,2)`, `(3,0)`, `(2,1)`, `(1,2)`, `(0,3)`, matching the Python implementation discussed earlier.


In [ ]:
using Pkg

function find_public_root(start_dirs)
    for start_dir in start_dirs
        dir = abspath(start_dir)
        while true
            if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "PulsePropagation.jl"))
                return dir
            end
            parent = dirname(dir)
            parent == dir && break
            dir = parent
        end
    end
    error("Could not find PulsePropagation public root")
end

const ROOT = find_public_root((pwd(), @__DIR__))
Pkg.activate(ROOT)


In [ ]:
using DelimitedFiles
using FFTW
using LinearAlgebra
using PyPlot
using Printf
using Random
using Statistics

isdefined(Main, :PulsePropagation) ||
    include(joinpath(ROOT, "src", "PulsePropagation.jl"))
using .PulsePropagation


## Parameters

The transverse analytic model uses the parabolic-GRIN field radius

`w0^2 = 2a / (k0 NA)`.

The propagation constants are loaded from the scalar GIF625 CSV table so that this notebook stays aligned with `gif625_exercises.ipynb` for dispersion while replacing the mode profiles and nonlinear tensor with the analytic HG construction.


In [ ]:
const NUM_MODES = 15
const NT = 2^13
const TIME_WINDOW_PS = 10.0
const PULSE_FWHM_PS = 0.150
const PULSE_ENERGY_NJ = 5.0
const PROP_LENGTH_M = 0.25 # ORIGINALLY 1.0
const PROP_SAVE_DZ_M = 0.5
const LAMBDA0_M = 1550e-9
const N2_M2_PER_W = 2.3e-20

const CORE_RADIUS_UM = 62.5 / 2
const NA = 0.275
const W0_UM = sqrt(2 * CORE_RADIUS_UM / ((2pi / (LAMBDA0_M * 1e6)) * NA))
const W0_M = W0_UM * 1e-6

const HG_ORDERS = [(0,0), (1,0), (0,1), (2,0), (1,1), (0,2), (3,0), (2,1), (1,2), (0,3), (4,0), (3,1), (2,2), (1,3), (0,4)]
const MODE_GROUPS = [1, 2, 2, 3, 3, 3, 4, 4, 4, 4,5,5,5,5,5]

fiber_dir = joinpath(ROOT, "GRIN_62.5um_wavelength1550nm_csv")
@assert isdir(fiber_dir)

modes = 1:NUM_MODES
(
    w0_um=W0_UM,
    pi_w0_squared_um2=pi * W0_UM^2,
    propagation_length_m=PROP_LENGTH_M,
    pulse_energy_nJ=PULSE_ENERGY_NJ,
)


## Analytic HG Modes

In [ ]:
function Hn_phys(n::Integer, x)
    n == 0 && return one(x)
    n == 1 && return 2x
    hm1 = one(x)
    h = 2x
    for k in 1:(n - 1)
        hp = 2x*h - 2k*hm1
        hm1 = h
        h = hp
    end
    return h
end

function hg_phi1d(n::Integer, x; w0=W0_M)
    norm_factor = (2/(pi*w0^2))^(1/4) / sqrt(2.0^n * factorial(n))
    return norm_factor * Hn_phys(n, sqrt(2)*x/w0) * exp(-(x/w0)^2)
end

# Plotting grid, in SI for normalization and um for axes.
mode_dx_um = 0.25
mode_x_um = collect((-400 ÷ 2):(400 ÷ 2 - 1)) .* mode_dx_um
mode_y_um = copy(mode_x_um)
mode_x_m = mode_x_um .* 1e-6
mode_y_m = mode_y_um .* 1e-6
dA_m2 = (mode_dx_um * 1e-6)^2

mode_profiles = zeros(Float64, length(mode_y_m), length(mode_x_m), NUM_MODES)
for (m, (nx_order, ny_order)) in enumerate(HG_ORDERS)
    profile = [hg_phi1d(nx_order, x) * hg_phi1d(ny_order, y) for y in mode_y_m, x in mode_x_m]
    profile ./= sqrt(sum(abs2, profile) * dA_m2)
    mode_profiles[:, :, m] .= profile
end

mode_norms = [sum(abs2, mode_profiles[:, :, m]) * dA_m2 for m in modes]
mode_overlap = [sum(mode_profiles[:, :, i] .* mode_profiles[:, :, j]) * dA_m2 for i in modes, j in modes]
(
    mode_profile_size=size(mode_profiles),
    norm_error=maximum(abs.(mode_norms .- 1)),
    orthogonality_error=norm(mode_overlap - I(NUM_MODES)) / norm(I(NUM_MODES)),
)


In [ ]:
fig, axs = subplots(3, 5; figsize=(14, 6), constrained_layout=true)
vmax = maximum(abs, mode_profiles[:, :, 1:NUM_MODES])
last_image = nothing

for m in modes
    row = div(m - 1, 5) + 1
    col = (m - 1) % 5 + 1
    ax = axs[row, col]
    global last_image = ax.imshow(
        mode_profiles[:, :, m];
        origin="lower",
        extent=(first(mode_x_um), last(mode_x_um), first(mode_y_um), last(mode_y_um)),
        cmap="RdBu_r",
        vmin=-vmax,
        vmax=vmax,
        interpolation="nearest",
    )
    ax.set_title("mode $m: $(HG_ORDERS[m])")
    ax.set_xlabel("x (um)")
    ax.set_ylabel("y (um)")
    ax.set_aspect("equal")
end

fig.colorbar(last_image, ax=axs[:], shrink=0.82, label="F (1/m)")
fig


## Analytic S Tensor

This section mirrors the Python implementation: use 12-point Gauss-Hermite quadrature with `xnode = (w0/2) u` and effective weights `Weff = (w0/2) wgh exp(u^2)`, then build

`S[p,l,m,n] = int dx dy F_p F_l F_m F_n`.

For real HG modes this is the same as the scalar Kerr overlap convention. The resulting tensor has units `1/m^2`.


In [ ]:
function hermgauss(n::Integer)
    beta = sqrt.((1:n-1) ./ 2)
    J = SymTridiagonal(zeros(n), beta)
    E = eigen(J)
    nodes = E.values
    weights = sqrt(pi) .* abs2.(E.vectors[1, :])
    return nodes, weights
end

function I4_1d(a::Integer, b::Integer, c::Integer, d::Integer; nGH=12, w0=W0_M)
    u, wgh = hermgauss(nGH)
    xnode = (w0 / 2) .* u
    Weff = (w0 / 2) .* wgh .* exp.(u.^2)
    return sum(Weff .* hg_phi1d.(a, xnode; w0=w0) .* hg_phi1d.(b, xnode; w0=w0) .* hg_phi1d.(c, xnode; w0=w0) .* hg_phi1d.(d, xnode; w0=w0))
end

I4_cache = Dict{NTuple{4, Int}, Float64}()
for a in 0:3, b in 0:3, c in 0:3, d in 0:3
    I4_cache[(a,b,c,d)] = I4_1d(a,b,c,d)
end

S_analytic = zeros(Float64, NUM_MODES, NUM_MODES, NUM_MODES, NUM_MODES)
for p in modes, l in modes, m in modes, n in modes
    px, py = HG_ORDERS[p]
    lx, ly = HG_ORDERS[l]
    mx, my = HG_ORDERS[m]
    nx, ny = HG_ORDERS[n]
    S_analytic[p,l,m,n] = I4_cache[(px,lx,mx,nx)] * I4_cache[(py,ly,my,ny)]
end

# Independent direct pixel-grid check of the same tensor.
S_pixel = zeros(Float64, NUM_MODES, NUM_MODES, NUM_MODES, NUM_MODES)
for p in modes, l in modes, m in modes, n in modes
    S_pixel[p,l,m,n] = sum(mode_profiles[:, :, p] .* mode_profiles[:, :, l] .* mode_profiles[:, :, m] .* mode_profiles[:, :, n]) * dA_m2
end

tensor_norm(A) = sqrt(sum(abs2, A))
S_relative_grid_error = tensor_norm(S_pixel - S_analytic) / tensor_norm(S_analytic)
(
    S1111=S_analytic[1,1,1,1],
    Aeff_um2=1/S_analytic[1,1,1,1] * 1e12,
    expected_Aeff_um2=pi * W0_UM^2,
    nonzero_entries=count(abs.(S_analytic) .> 1e-30),
    relative_pixel_grid_error=S_relative_grid_error,
)


In [ ]:
mode_profiles

In [ ]:
figure(figsize=(4,3))
vmax = maximum(abs, mode_profiles[:, :, 1])
last_image = nothing

pcolormesh(mode_x_um, mode_y_um, mode_profiles[:,:,13],cmap="RdBu_r",vmin=-vmax,vmax=vmax)
# xlim(-1,1)
# ylim(-1,1)
colorbar()
display(gcf())


In [ ]:
dx = diff(mode_x_um)[1]*1e-6;
dy = diff(mode_y_um)[1]*1e-6;
dA = dx*dy;
ind_0 = 201;
mode_x_um[ind_0], mode_y_um[ind_0] ## coordinate of x, y = 0
## Coordinates in x: -0.25 -> 0 -> 0.25. Each inherits a square of side length 250 nm.

In [ ]:
mode_x_um[199:202]

In [ ]:
## Normalization is sum dA |ϕ|^2 <-> ∫ dA |ϕ|^2 = 1.

[sum(mode_profiles[:, :, ii] .* mode_profiles[:, :, ii])*dA  for ii = 1:NUM_MODES]

In [ ]:
[abs2.(mode_profiles[ind_0, ind_0, ii]) for ii = 1:NUM_MODES]

In [ ]:
ϕ_0_squared = [abs2.(mode_profiles[ind_0, ind_0, ii]) for ii = 1:NUM_MODES]

In [ ]:
## Final answer:
intensity_gain = 1
fano_in = 2*intensity_gain - 1; ##Fano of initial state is 2G - 1. All modes have this Fano, no corr.
fano_filter = fano_in * (1 - dA*sum(ϕ_0_squared))

In [ ]:
fano_filter_groups_1 = fano_in * (1 - dA*sum(ϕ_0_squared[1])) # 
fano_filter_groups_123 = fano_in * (1 - dA*sum(ϕ_0_squared[[1;4;6]]))
fano_filter_groups_12345 = fano_in * (1 - dA*sum(ϕ_0_squared[[1;4;6;11;13;15]]))
fano_filter_groups_all = fano_in * (1 - dA*sum(ϕ_0_squared));
#(1 .- [fano_filter_groups_1 , fano_filter_groups_123 , fano_filter_groups_12345, fano_filter_groups_all])
2000*[fano_filter_groups_1 , fano_filter_groups_123 , fano_filter_groups_12345, fano_filter_groups_all]

In [ ]:
[fano_filter_groups_1 , fano_filter_groups_123 , fano_filter_groups_12345, fano_filter_groups_all]

## Comparison To The Python HG Implementation

In [ ]:
python_expected_ratios = [1.0, 0.5, 0.5, 0.375, 0.25, 0.375, 0.3125, 0.1875, 0.1875, 0.3125]
computed_ratios = [S_analytic[p,p,1,1] / S_analytic[1,1,1,1] for p in modes]
python_comparison_table = hcat(collect(modes), [o[1] for o in HG_ORDERS], [o[2] for o in HG_ORDERS], computed_ratios, python_expected_ratios, computed_ratios .- python_expected_ratios)

python_equivalence_summary = (
    w0_m = W0_M,
    Q0000_1_per_m2 = S_analytic[1,1,1,1],
    expected_Q0000_1_per_m2 = 1/(pi * W0_M^2),
    Q0000_relative_error = (S_analytic[1,1,1,1] - 1/(pi * W0_M^2)) / (1/(pi * W0_M^2)),
    max_ratio_error = maximum(abs.(computed_ratios .- python_expected_ratios)),
    nonzero_Q_entries = count(abs.(S_analytic) .> 1e-30),
)

println("columns: mode, nx, ny, computed Q[p,p,0,0]/Q[0,0,0,0], Python expected, difference")
display(python_comparison_table)
python_equivalence_summary


## Dispersion And Propagation Problem

In [ ]:
function public_beta_units_to_ps_per_m_csv(betas)
    size(betas, 1) > 8 && return Array{Float64}(betas)
    scale = reshape(0.001 .^ collect(-1:size(betas, 1)-2), :, 1)
    return Array{Float64}(betas .* scale)
end

raw_betas = readdlm(joinpath(fiber_dir, "betas.csv"), ',', Float64)
betas = public_beta_units_to_ps_per_m_csv(raw_betas)[:, modes]
betas[6,:] .= 0

analytic_system = PassiveFiber(
    length=PROP_LENGTH_M,
    lambda0=LAMBDA0_M,
    f0=2.99792458e-4 / LAMBDA0_M,
    material=Silica(),
    betas=betas,
    sr=S_analytic,
    n2=N2_M2_PER_W,
    raman_fraction=0.0,
    source_path="analytic-HG GIF625 basis with public scalar betas",
)
analytic_dofs = ModalField(modes)
prop_grid = TimeGrid(Nt=NT, window=TIME_WINDOW_PS)

modal_coefficients = ones(ComplexF64, NUM_MODES) ./ sqrt(NUM_MODES)
prop_fields = gaussian_pulse(
    prop_grid;
    fwhm=PULSE_FWHM_PS,
    total_energy=PULSE_ENERGY_NJ,
    dofs=analytic_dofs,
    coefficients=modal_coefficients,
)

input_modal_energy_nJ = vec(sum(abs2, prop_fields[:, :, 1]; dims=1)) .* prop_grid.dt ./ 1e3
(
    beta_size=size(betas),
    input_total_energy_nJ=sum(input_modal_energy_nJ),
    input_modal_energy_nJ=input_modal_energy_nJ,
)


In [ ]:
betas

In [ ]:
prop_zsave = collect(0.0:PROP_SAVE_DZ_M:PROP_LENGTH_M)
prop_zsave[end] == PROP_LENGTH_M || push!(prop_zsave, PROP_LENGTH_M)

prop_compression = CPCompression(
    target_error=1e-5,
    initial_rank=64,
    max_rank=1024,
    rank_growth=2,
    maxiter=300,
    tol=1e-9,
    ridge=1e-8,
    check_every=10,
    seed=11,
    verbose=false,
)

prop_solver = RK4IPSolver(
    stepping=FixedStep(dz=2.5e-4),
    saveat=SaveAt(prop_zsave),
    compression=prop_compression,
    cp_rhs=CachedCPRHS(; fftw_threads=1, fftw_flags=:MEASURE),
    reltol=1e-8,
    abstol=1e-8,
)

prop_problem = PulsePropagationProblem(
    grid=prop_grid,
    system=analytic_system,
    dofs=analytic_dofs,
    terms=PropagationTerms(Dispersion(), Kerr()),
    solver=prop_solver,
    fields=prop_fields,
)

prop_problem


## Solve

In [ ]:
prop_solution = solve(prop_problem)
prop_cp = compressed_tensor(prop_solution)

solve_summary = (
    propagation_length_m=PROP_LENGTH_M,
    fwhm_ps=PULSE_FWHM_PS,
    input_energy_nJ=sum(input_modal_energy_nJ),
    saved_z_m=prop_solution.output.z,
    cp_rank=compression_rank(prop_solution),
    cp_error=compression_error(prop_solution),
)
solve_summary


## Final Spectrum And Modal Energy

In [ ]:
prop_freq_THz = frequency_axis(prop_grid; shifted=true)
prop_final_spectrum = spectrum(prop_solution; z=:final, shifted=true, sum_modes=false)
prop_final_total_spectrum = vec(sum(prop_final_spectrum; dims=2))
prop_spectrum_norm = maximum(prop_final_total_spectrum)

fig, ax = subplots(1, 1; figsize=(7, 4.5), constrained_layout=true)
ax.plot(prop_freq_THz, prop_final_total_spectrum ./ prop_spectrum_norm; color="black", linewidth=2, label="total")
for m in modes
    mode_spectrum = prop_final_spectrum[:, m]
    ax.plot(prop_freq_THz, mode_spectrum ./ prop_spectrum_norm; linewidth=0.9, alpha=0.65, label="mode $m")
end
ax.set_xlabel("frequency offset (THz)")
ax.set_ylabel("normalized spectral power")
ax.set_title("Analytic HG final spectrum")
ax.grid(true; alpha=0.25)
ax.set_xlim(-10, 10)
ax.legend(ncol=2, fontsize=8)
fig


In [ ]:
final_modal_energy_nJ = modal_energy(prop_solution; z=:final)
mode_group_energy_nJ = [sum(final_modal_energy_nJ[MODE_GROUPS .== g]) for g in 1:4]
energy_summary = (
    final_total_energy_nJ=sum(final_modal_energy_nJ),
    final_modal_energy_nJ=final_modal_energy_nJ,
    final_mode_group_energy_nJ=mode_group_energy_nJ,
    relative_energy_error=(sum(final_modal_energy_nJ) - sum(input_modal_energy_nJ)) / sum(input_modal_energy_nJ),
)
energy_summary


In [ ]:
mode_group_energy_nJ

In [ ]:
sum(final_modal_energy_nJ)

In [ ]:
mode_groups = [1:1, 2:3, 4:6, 7:10]
mode_group_names = ["1", "2:3", "4:6", "7:10"]

final_modal_energy_nJ = modal_energy(prop_solution; z=:final)
mode_group_energy_nJ = [sum(final_modal_energy_nJ[group]) for group in mode_groups]
highest_group_index = argmax(mode_group_energy_nJ)
highest_energy_group = mode_groups[highest_group_index]

# # For this launch, the highest-energy group is expected to be modes 7:10.
# expected_highest_group = 7:10
# @assert collect(highest_energy_group) == collect(expected_highest_group) "Highest-energy group was $(mode_group_names[highest_group_index]), not 7:10."

nt_group = size(field(prop_solution; z=:final, domain=:time), 1)
full_spectrum_filter = ones(Float64, nt_group)

mode_group_photon_observable = SpectralPhotonNumber(
    filter=full_spectrum_filter,
    modes=highest_energy_group,
    shifted=false,
)

In [ ]:
mode_group_variance_objective = VarianceObjective(
    mode_group_photon_observable;
    method=PulsePropagation.Adjoint(),
    normalization=PhotonNormalized(),
    raman=:off,
    reltol=1e-6,
    abstol=1e-6,
    return_lambdaw_zsave=false,
)

mode_group_fluctuations = variance_result(
    mode_group_variance_objective,
    prop_problem,
    prop_solution,
)

In [ ]:
(
    selected_mode_group=mode_group_names[highest_group_index],
    selected_modes=collect(highest_energy_group),
    final_modal_energy_nJ=final_modal_energy_nJ,
    mode_group_energy_nJ=mode_group_energy_nJ,
    photon_number=mode_group_fluctuations.value,
    photon_number_variance=mode_group_fluctuations.variance,
    fano_factor=mode_group_fluctuations.fano,
)


In [ ]:
mode_group_fluctuations.variance # 4.3e10

In [ ]:
# 1.914e-9 / (1.05e-34 * 1.52e15 * 1240/1550)

In [ ]:
# Multimode shot-noise Monte Carlo check for the highest-energy mode group.
# This keeps the CP-compressed solve path used above while using the public
# PulsePropagation Fourier and photon-noise helpers directly.

STOCHASTIC_NTRAJ = 500
STOCHASTIC_SEED = 20260616
stochastic_noise = ShotNoise(; noise_scale=0.5)
stochastic_rng = MersenneTwister(STOCHASTIC_SEED)
stochastic_objs = backend_objects(prop_problem)

stochastic_uω0 = PulsePropagation.inverse_fft(stochastic_objs.ic.fields[:, :, end], dims=1)
stochastic_uω_ensemble = PulsePropagation.photon_vacuum_ensemble(
    stochastic_uω0,
    stochastic_objs.ic.dt,
    stochastic_objs.sim.f0,
    STOCHASTIC_NTRAJ;
    rng=stochastic_rng,
    noise_scale=Float64(stochastic_noise.noise_scale),
)

stochastic_mode_group_filter = zeros(Float64, prop_grid.nt, NUM_MODES)
stochastic_mode_group_filter[:, collect(highest_energy_group)] .= 1.0

# Reuse the CP tensor fitted by prop_solution so the stochastic trajectories do
# not refit the analytic S tensor each time. Save only z=0 and z=L.
stochastic_cp_solver = RK4IPSolver(
    stepping=prop_solver.stepping,
    saveat=SaveAt([0.0, PROP_LENGTH_M]),
    compression=CPCompression(
        tensor=compressed_tensor(prop_solution),
        target_error=compression_error(prop_solution),
    ),
    cp_rhs=CachedCPRHS(; fftw_threads=1, fftw_flags=:ESTIMATE),
    reltol=prop_solver.reltol,
    abstol=prop_solver.abstol,
)

stochastic_photon_samples = zeros(Float64, STOCHASTIC_NTRAJ)

for q in 1:STOCHASTIC_NTRAJ
    fields_q = reshape(
        PulsePropagation.forward_fft(stochastic_uω_ensemble[:, :, q], dims=1),
        prop_grid.nt,
        NUM_MODES,
        1,
    )
    problem_q = PulsePropagationProblem(
        grid=prop_grid,
        system=analytic_system,
        dofs=analytic_dofs,
        terms=PropagationTerms(Dispersion(), Kerr()),
        solver=stochastic_cp_solver,
        fields=fields_q,
    )
    sol_q = solve(problem_q)
    stochastic_photon_samples[q] = photon_number(
        sol_q,
        stochastic_mode_group_filter;
        z=:final,
        shifted=false,
    )
end

stochastic_mean_photon_number = mean(stochastic_photon_samples)
stochastic_photon_number_variance = var(stochastic_photon_samples; corrected=true)
stochastic_fano_factor = stochastic_photon_number_variance / stochastic_mean_photon_number
stochastic_variance_standard_error = sqrt(2 / (STOCHASTIC_NTRAJ - 1)) * mode_group_fluctuations.variance
stochastic_variance_z_score = (stochastic_photon_number_variance - mode_group_fluctuations.variance) / stochastic_variance_standard_error

(
    ntraj=STOCHASTIC_NTRAJ,
    seed=STOCHASTIC_SEED,
    selected_mode_group=mode_group_names[highest_group_index],
    selected_modes=collect(highest_energy_group),
    photon_samples=stochastic_photon_samples,
    mean_photon_number=stochastic_mean_photon_number,
    photon_number_variance=stochastic_photon_number_variance,
    fano_factor=stochastic_fano_factor,
    variance_z_score=stochastic_variance_z_score,
    adjoint_photon_number=mode_group_fluctuations.value,
    adjoint_photon_number_variance=mode_group_fluctuations.variance,
    adjoint_fano_factor=mode_group_fluctuations.fano,
)


In [ ]:
# mean_photon_number = 1.7821199281292545e10, photon_number_variance = 3.663510144529604e10, fano_factor = 2.055703483645628 250 traj